# **EDA Notebook**



---
## 0. Setup Environment

In [4]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 62.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [5]:
group_name = "Group 24"
student_name = "Mukesh Murugesan"
student_id = "25747763"

In [6]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [7]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [8]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [9]:
import pandas as pd
import altair as alt
import numpy as np
import matplotlib.pyplot as plt

---
## B. Data Understanding

In [10]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "person.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [11]:
# Shape: how many rows and columns
print("Shape:", df.shape)
print()

# Column names and data types
print("Data Types:")
print(df.dtypes)
print()

# First 5 rows
df.head()

Shape: (25163, 10)

Data Types:
person_id                   object
name_style                 float64
email_promotion            float64
person_type                 object
title                       object
first_name                  object
middle_name                 object
last_name                   object
suffix                      object
additional_contact_info     object
dtype: object



,person_id,name_style,email_promotion,person_type,title,first_name,middle_name,last_name,suffix,additional_contact_info
0,0e7956c1-835b-487b-b4b3-b83d08811282,0.0,0.0,IN,NaN,Calvin,A,Sharma,NaN,NaN
1,333a1b3e-04c5-4c0f-80db-d840f5cc6593,0.0,1.0,SC,Mr.,John,Y.,Chen,NaN,NaN
2,ddfd0d6e-1be4-4c5a-9eaf-6bce0d98c302,0.0,2.0,IN,NaN,Kristine,K,Moreno,NaN,NaN
3,aef31e4a-052f-47dc-85bd-31b3312a09c9,0.0,0.0,SC,Mr.,Eddie,M.,Holmes,NaN,NaN
4,63635d45-76e1-4f06-a25f-424bee9af615,0.0,0.0,VC,Mr.,Richard,T.,Young,NaN,NaN


In [12]:
# ── OBSERVATION SUMMARY TABLE ───────────────────────────────────

summary = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
    'Missing Count': df.isnull().sum().values,
    'Missing %': (df.isnull().sum().values / len(df) * 100).round(2),
    'Unique Values': [df[c].nunique() for c in df.columns],
    'Recommended Action': [
        'Use as join key (note: not fully unique)',    # person_id
        'Impute mode(0), then drop — all same value',  # name_style
        'Impute mode(0), one-hot encode',              # email_promotion
        'Impute mode(IN), one-hot encode',             # person_type
        'DROP — 94.9% missing',                        # title
        'Use as-is',                                   # first_name
        'Engineer → has_middle_name binary flag',      # middle_name
        'Use as-is',                                   # last_name
        'DROP — 99.7% missing',                        # suffix
        'DROP — 99.9% missing',                        # additional_contact_info
    ]
})
print(summary.to_string(index=False))

                 Column Data Type  Missing Count  Missing %  Unique Values                         Recommended Action
              person_id    object              0       0.00          19972   Use as join key (note: not fully unique)
             name_style   float64             73       0.29              1 Impute mode(0), then drop — all same value
        email_promotion   float64            210       0.83              3             Impute mode(0), one-hot encode
            person_type    object             35       0.14              6            Impute mode(IN), one-hot encode
                  title    object          23888      94.93              6                       DROP — 94.9% missing
             first_name    object              0       0.00           1018                                  Use as-is
            middle_name    object          10757      42.75             71     Engineer → has_middle_name binary flag
              last_name    object              0       0

In [13]:
# ── MISSING VALUES
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
})
print("Missing Values per Column:")
print(missing_df)
print()

# ── DUPLICATE INVESTIGATION
# person_id should uniquely identify a person — check if it does
total_rows = len(df)
unique_ids = df['person_id'].nunique()
duplicate_rows = df.duplicated().sum()

print(f"Total rows:           {total_rows}")
print(f"Unique person_id:     {unique_ids}")
print(f"Difference:           {total_rows - unique_ids}  ← same person_id appears in multiple rows")
print(f"Fully duplicate rows: {duplicate_rows}")
print()

# Investigate: are duplicates the same person_type or different?
dup_mask = df.duplicated('person_id', keep=False)
dup_df = df[dup_mask]
print("person_type breakdown among duplicate person_ids:")
print(dup_df['person_type'].value_counts())
print()
print("Example — same person_id appearing more than once:")
print(df[df['person_id'] == df['person_id'].value_counts().index[0]])

Missing Values per Column:
                         Missing Count  Missing %
person_id                            0       0.00
name_style                          73       0.29
email_promotion                    210       0.83
person_type                         35       0.14
title                            23888      94.93
first_name                           0       0.00
middle_name                      10757      42.75
last_name                            0       0.00
suffix                           25098      99.74
additional_contact_info          25152      99.96

Total rows:           25163
Unique person_id:     19972
Difference:           5191  ← same person_id appears in multiple rows
Fully duplicate rows: 5191

person_type breakdown among duplicate person_ids:
person_type
IN    9179
SC     386
EM     165
GC     156
VC      66
SP      10
Name: count, dtype: int64

Example — same person_id appearing more than once:
                                  person_id  name_style  email

In [14]:
dataset_insights = """
The person dataset contains 25,163 records across 10 columns, capturing demographic
and contact information for individuals linked to the retailer's business operations.

Dimensions: 25,163 rows x 10 columns.

Columns overview:
  - person_id: Identifier for each person (UUID string). NOTE: not fully unique —
    19,972 unique values across 25,163 rows, meaning some persons appear multiple times.
  - person_type: Role of the person (IN, SC, GC, EM, VC, SP).
  - name_style, first_name, middle_name, last_name, suffix, title: Name fields.
  - email_promotion: Marketing preference (0=none, 1=AW only, 2=partner promotions).
  - additional_contact_info: XML contact data — nearly entirely missing.

Missing data summary:
  - additional_contact_info: 99.9% missing — DROP this column.
  - suffix: 99.7% missing — DROP this column.
  - title: 94.9% missing — DROP this column.
  - middle_name: 42.7% missing — ENGINEER as binary flag has_middle_name.
  - email_promotion: 0.83% missing — IMPUTE with mode (0).
  - name_style: 0.29% missing — IMPUTE with mode (0).
  - person_type: 0.14% missing — IMPUTE with mode (IN) or drop rows.

The dominant person type is IN (Individual customers) at 92.3% of all records.
This dataset can be joined with sales data to enrich customer-level features.
"""

In [15]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

In [16]:
# ── BUSINESS CONTEXT & HYPOTHESIS ──────────────────────────────
# Use Case: Predict the total sales amount (line_total) for each order line item.
#
# As a new data scientist for a global bike retailer, understanding WHO the
# customers are and HOW they differ is the foundation of any predictive model.
#
# Hypothesis: Customer demographic attributes (person_type, email_promotion,
# has_middle_name) derived from the person dataset can be used as enrichment
# features to improve regression model accuracy for predicting line_total.
#
# This EDA investigates the person dataset to:
#   1. Understand data quality (missing values, duplicates)
#   2. Identify usable features for joining with sales data
#   3. Detect anomalies that could harm model performance
print("Business context loaded. Target: predict line_total using enriched customer features.")

Business context loaded. Target: predict line_total using enriched customer features.


### B.2 Explore Feature of Interest `\<person_type\>`

In [17]:
# Frequency table
print("person_type counts:")
print(df['person_type'].value_counts())
print()
print("Proportions (%):")
print((df['person_type'].value_counts(normalize=True) * 100).round(2))

person_type counts:
person_type
IN    23229
SC      955
GC      371
EM      359
VC      192
SP       22
Name: count, dtype: int64

Proportions (%):
person_type
IN    92.44
SC     3.80
GC     1.48
EM     1.43
VC     0.76
SP     0.09
Name: proportion, dtype: float64


In [18]:
# Bar chart of person_type distribution
pt_counts = df['person_type'].value_counts().reset_index()
pt_counts.columns = ['person_type', 'count']

chart1 = alt.Chart(pt_counts).mark_bar(color='steelblue').encode(
    x=alt.X('person_type:N', sort='-y', title='Person Type'),
    y=alt.Y('count:Q', title='Count'),
    tooltip=['person_type', 'count']
).properties(
    title='Distribution of Person Type',
    width=400, height=300
)
chart1

alt.Chart(...)

In [19]:
feature_1_insights = """
Feature: person_type

person_type categorises each record by their business relationship with the retailer.
There are 6 unique values:
  - IN  (Individual):     23,229 — 92.3%  ← primary customer group
  - SC  (Store Contact):     955 —  3.8%
  - GC  (General Contact):   371 —  1.5%
  - EM  (Employee):          359 —  1.4%
  - VC  (Vendor Contact):    192 —  0.8%
  - SP  (Sales Person):       22 —  0.1%

Missing values: 35 rows (0.14%) — safe to impute with mode 'IN' or drop.

Issues:
  - Severe class imbalance: IN accounts for 92.3% of data. Non-IN types provide
    very little signal. For modelling, this will be one-hot encoded, and analysis
    will be focused on IN individuals.

Business relevance:
  Filtering to IN (individual customers) gives us a clean customer base for
  predicting sales line totals. Other types (employees, vendors) behave differently
  and may distort the model if included.
"""

In [20]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `\<email_promotion\>`

In [21]:
# Frequency and proportion
print("email_promotion counts:")
print(df['email_promotion'].value_counts().sort_index())
print()
print("Proportions (%):")
print((df['email_promotion'].value_counts(normalize=True).sort_index() * 100).round(2))
print()
print("Missing values:", df['email_promotion'].isnull().sum())

email_promotion counts:
email_promotion
0.0    13936
1.0     6292
2.0     4725
Name: count, dtype: int64

Proportions (%):
email_promotion
0.0    55.85
1.0    25.22
2.0    18.94
Name: proportion, dtype: float64

Missing values: 210


In [22]:
# Bar chart
ep_counts = df['email_promotion'].value_counts().reset_index()
ep_counts.columns = ['email_promotion', 'count']
ep_counts['email_promotion'] = ep_counts['email_promotion'].astype(str)

chart2 = alt.Chart(ep_counts).mark_bar(color='coral').encode(
    x=alt.X('email_promotion:N',
            sort=['0.0','1.0','2.0'],
            title='Email Promotion (0=None, 1=AW Only, 2=Partner)'),
    y=alt.Y('count:Q', title='Count'),
    tooltip=['email_promotion', 'count']
).properties(
    title='Distribution of Email Promotion Preference',
    width=400, height=300
)
chart2

alt.Chart(...)

In [23]:
# ── CROSS-FEATURE ANALYSIS: person_type vs email_promotion ──────
# HD requires quantitative analysis to support decisions.
# Does email promotion preference differ by person type?

crosstab = pd.crosstab(df['person_type'], df['email_promotion'],
                        normalize='index').round(3) * 100
crosstab.columns = ['No Promo (0)', 'AW Only (1)', 'Partner (2)']
print("Email Promotion breakdown by Person Type (%):")
print(crosstab)
print()
print("Insight: IN (individual customers) follow a similar opt-out pattern (~55%)")
print("to the overall dataset, confirming they drive the overall distribution.")

Email Promotion breakdown by Person Type (%):
             No Promo (0)  AW Only (1)  Partner (2)
person_type                                        
EM                   55.3         21.3         23.3
GC                   55.2         28.8         15.9
IN                   55.9         25.1         19.0
SC                   55.6         25.6         18.8
SP                   57.1         33.3          9.5
VC                   54.5         30.4         15.2

Insight: IN (individual customers) follow a similar opt-out pattern (~55%)
to the overall dataset, confirming they drive the overall distribution.


In [24]:
feature_2_insights = """
Feature: email_promotion

email_promotion records each person's consent level for receiving promotional emails:
  0 = No promotional emails:         13,936 (55.4%)
  1 = Adventure Works promos only:    6,292 (25.0%)
  2 = Partner promotional emails:     4,725 (18.8%)

Missing values: 210 (0.83%) — impute with mode (0).

Distribution:
  - More than half of customers (55.4%) have opted out of email marketing.
  - About 44% are reachable via email promotions — a valuable marketing segment.

Issues:
  - Although stored as a float, this is an ordinal categorical variable.
    It should NOT be treated as continuous. Will be one-hot encoded for modelling.
  - 210 missing values — small enough to safely impute with mode (0).

Business relevance:
  Customers who opt into promotions (levels 1 and 2) are likely more
  price/offer-sensitive and may respond differently to discounts, potentially
  ordering higher quantities. This feature adds customer behaviour context
  to the sales prediction model.
"""

In [25]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `\<missing data pattern\>`

In [26]:
# Missing % per column — visualised
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_viz = missing_pct.reset_index()
missing_viz.columns = ['column', 'missing_pct']

chart3 = alt.Chart(missing_viz).mark_bar(color='tomato').encode(
    x=alt.X('missing_pct:Q', title='Missing (%)'),
    y=alt.Y('column:N', sort='-x', title='Column'),
    tooltip=['column', 'missing_pct']
).properties(
    title='Missing Value % per Column — person dataset',
    width=450, height=300
)
chart3

alt.Chart(...)

In [27]:
print("Columns to DROP (>50% missing):")
print(missing_pct[missing_pct > 50])
print()
print("Columns to ENGINEER (middle_name → binary flag):")
print(missing_pct[(missing_pct > 5) & (missing_pct <= 50)])
print()
print("Columns to IMPUTE or USE AS-IS (<5% missing):")
print(missing_pct[missing_pct <= 5])

Columns to DROP (>50% missing):
additional_contact_info    99.956285
suffix                     99.741684
title                      94.933037
dtype: float64

Columns to ENGINEER (middle_name → binary flag):
middle_name    42.749275
dtype: float64

Columns to IMPUTE or USE AS-IS (<5% missing):
email_promotion    0.834559
name_style         0.290108
person_type        0.139093
person_id          0.000000
first_name         0.000000
last_name          0.000000
dtype: float64


In [28]:
feature_n_insights = """
Feature Analysis: Missing Data Pattern across the person dataset

Reviewing missing values is essential before building any ML model.
Each column is assessed and a strategy is assigned:

DROP (>90% missing — no useful signal):
  - additional_contact_info: 99.9% missing
  - suffix:                  99.7% missing
  - title:                   94.9% missing

ENGINEER (moderate missing — extracting partial signal):
  - middle_name: 42.7% missing. Rather than imputing fabricated names,
    create a binary feature: has_middle_name = 1 if present, else 0.
    This preserves the information that a name is recorded without guessing values.

IMPUTE with MODE (small missingness — safe to fill):
  - email_promotion: 0.83% missing → fill with 0 (most common)
  - name_style:      0.29% missing → fill with 0.0 (most common)
  - person_type:     0.14% missing → fill with 'IN' (most common)

USE AS-IS (no missing values):
  - person_id, first_name, last_name

After applying these strategies, the person dataset yields clean,
usable features for joining with sales order data in the preparation notebook.
"""

In [29]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)